In [0]:
from pyspark.sql.types import *

data = [
    (1, ("Mahesh","Vemula"), 3.5, ["MRA","EMMERT","DDA","SAS","PICA"], "Hyderabad"),
    (2, ("Gopal","Cheleboyana"), 3.0, ["MRA","EMMERT","DDA","PICA"], "Hyderabad"),
    (3, ("Teja","Gembelli"), 4.0, ["MRA","Compass"], "Hyderabad"),
    (4, ("Boopathi","Thagarasu"), 12.0, ["MRA","NLP","DDA"], "Chennai"),
    (5, ("Sarath","Dumpa"), 3.0, ["SAS","EMMERT","DDA"], "Noida"),
    (6, ("Divya",""), 5.0, ["BI","SAS"], "Hyderabad"),
    (7, ("Ravi","Kumar"), 8.0, ["Python","Spark"], "Bangalore"),
    (8, ("Lavanya","Reddy"), None, ["SQL","ADF"], None)
]

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StructType([
        StructField("first_name", StringType()),
        StructField("last_name", StringType())
    ])),
    StructField("experience", DoubleType()),
    StructField("team", ArrayType(StringType())),
    StructField("city", StringType())
])

df = spark.createDataFrame(data,schema)

display(df)

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("grade", when(col("experience") <= 4 ,"C")
                   .when(col("experience") <= 8 ,"B")
                   .otherwise("A"))

df.display()

In [0]:
display(df.select("name.first_name"))
display(df.filter(array_contains("team","Spark")))

In [0]:
display(df.filter(array_contains("team","Spark")))
# display(df)

In [0]:
display(df.withColumnRenamed("id","emp_id"))

In [0]:
# display(df.filter(col("grade") == "C"))
# display(df.filter((col("grade") == "C") | (col("grade") == "B")))
# display(df.filter(col("city").isin("Hyderabad")))
# display(df.filter(~col("city").isin("Hyderabad")))
# display(df.filter(col("city").contains("Che")))
# display(df.filter(~col("city").contains("Che")))
# display(df.filter(col("city").startswith("Ban")))
# display(df.filter(col("city").endswith("bad")))
# display(df.filter(col("id").between(1,3)))
# display(df.filter(col("city").like("%Ba%")))

In [0]:
# display(df.select("id","name.First_name","Grade").where(col("Grade") == "C"))
# display(df.filter(col("city").isNull()))
# display(df.filter(col("city").isNotNull()))

In [0]:
# display(df.na.fill("NA"))
# display(df.fillna("NA",["city","experience"]))

In [0]:
# display(df.groupBy("city").agg(count("*").alias("count")))
# display(df.select(substring("city",1,3).alias("substr")))
# display(df.orderBy(col("id").asc()))
# display(df.select(length("city").alias("length")))
# display(df.select(trim("city").alias("lengthtrm")))

In [0]:
target_data = [
    (1,"Mahesh","Hyderabad"),
    (2,"Gopal","Chennai"),
    (3,"Teja","Bangalore")
]

target_df = spark.createDataFrame(
    target_data,
    ["id","name","city"]
)

In [0]:
target_df.write.saveAsTable("employee_master")

In [0]:
%sql
select * from employee_master

In [0]:
source_data = [
    (1,"Mahesh","Pune"),
    (2,"Gopal","Chennai"),
    (4,"Divya","Hyderabad")
]

source_df = spark.createDataFrame(
    source_data,
    ["id","name","city"]
)


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "employee_master"
)

target.alias("trg") \
.merge(
    source_df.alias("src"),
    "trg.id = src.id"
) \
.whenMatchedUpdate(set={
    "name":"src.name",
    "city":"src.city"
}) \
.whenNotMatchedInsert(values={
    "id":"src.id",
    "name":"src.name",
    "city":"src.city"
}) \
.execute()

In [0]:
%sql
select * from employee_master

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark,"employee_master")
delta_table.update(condition = "id = 4", set = {"name": "'arjun'"})

In [0]:
%sql
MERGE INTO target t
USING src tt
ON t.id = tt.id

WHEN MATCHED THEN
  UPDATE SET
    t.name = tt.name,
    t.city = tt.city

WHEN NOT MATCHED THEN
  INSERT (
    id,
    name,
    city
  )
  VALUES (
    tt.id,
    tt.name,
    tt.city
  )

In [0]:
# display(df.select("city").distinct())
display(df)

In [0]:
df1 =df.select("id","name.first_name","name.last_name","experience",explode("team"),"city")
display(df1)

In [0]:
df2=df1.groupBy('id','first_name','last_name','experience','city').agg(collect_list('col').alias("team"))
# df2 = df2.select("id",concat_ws(",",col("team")).alias("team"))
# display(df2)

In [0]:
display(df.select('*',posexplode('team')))

In [0]:
dates = [
  (1, '2020-01-01', 'Jan-01-2020'),
  (2, '2020-01-02', 'Jan-02-2020'),
  (3, '2020-01-03', 'Jan-03-2020')]

dates_df = spark.createDataFrame(
    dates,
    "id int, date string, date_str string"
)

display(dates_df)

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, date_format

# display(dates_df.withColumn("dateformat", to_date(col("date"), "yyyy-MM-dd")))
# display(dates_df.withColumn("dateformat", to_timestamp(col("date"), "yyyy-MM-dd")))
display(dates_df.withColumn("dateformat", date_format(to_date(col("date_str"), "MMM-dd-yyyy"), "yyyy-MM-dd")))



In [0]:
data = [
    (1,"Mahesh","Hyderabad",10),
    (1,"Mahesh","Chennai",20),
    (1,"Mahesh","Noida",30),

    (2,"Gopal","Hyderabad",15),
    (2,"Gopal","Chennai",25),

    (3,"Teja","Hyderabad",35),
    (3,"Teja","Noida",45)
]

df = spark.createDataFrame(
    data,
    ["id","name","city","experience"]
)

display(df)

In [0]:
df.createOrReplaceTempView("employee")

In [0]:
%sql
create or replace temp view employee_pivot as
select * from employee
pivot(sum(experience) for city in ('Hyderabad', 'Chennai','Noida'))

In [0]:
%sql
select * from employee_pivot
unpivot(experience for city in (Hyderabad, Chennai, Noida))

In [0]:
# https://jsonplaceholder.typicode.com/posts

In [0]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts")
data = response.json()
# print(data)
# print(data[:2])

df = spark.createDataFrame(data)
display(df)